In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bootstrap = "pkc-7prvp.centralindia.azure.confluent.cloud:9092"  # your cluster
topic = "topic_two"

df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap) \
    .option("subscribe", topic) \
    .option("startingOffsets", "latest") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option(
        "kafka.sasl.jaas.config",
        f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
        f"required username='OO7RQD4EPIS23XEB' password='cfltE8YBRZuAEILd6Wz9bXZfa5SqhUPKA/HRzbHreeni8wVAdcQoDnGH9k2yucww';"
    ) \
    .load()
#    .option("kafka.session.timeout.ms", "45000") \
#    .option("kafka.request.timeout.ms", "60000") \


In [0]:
df = df_raw.select(col("value").cast("String"))

schema = StructType([
    StructField("source", StringType()),
    StructField("temperature", FloatType()),
    StructField("timestamp", DoubleType())
])

df = df.withColumn("json", from_json(col("value"), schema))

df = df.select(
    col("json.source").alias("topic_name"),
    col("json.temperature").alias("temperature"),
    col("json.timestamp").alias("timestamp")
)

df = df.withColumn("timestamp", from_unixtime(col("timestamp").cast("long")))

In [0]:
query = df.writeStream \
    .trigger(once=True) \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/practice/demo_data/kafka_practice") \
    .toTable("practice.demo_data.kafka")